# Q4 · How the geographical distribution changed
Three maps compare **equal nine-year periods** using the same boundaries, projection and colour scale. Darker red means a larger share of that period’s located fires. Percentages reveal geographical concentration even when overall counts change; they are **not fire risk, fires per km² or burned area**. Hover for comarca, count and share.

Unlike Q3’s change in counts, this answers where a larger or smaller fraction of Catalonia’s fires occurred. Sequential light-to-dark red signals greater fire concentration; lightness remains readable with colour-vision deficiencies. Red does not prove worsening risk: a share can rise even if the fire count falls. Adjacent maps enable comparison without remembering an earlier view. Individual fire dots were considered, but overlap would hide concentrations.

We use merged records from 1998–2024 and the exported comarca codes in `CleanData.csv`. Present-day comarca definitions are applied to all periods. Unknown locations are reported and excluded from percentage denominators. Large comarques occupy more map area; compare colour and tooltip values, not polygon size. The switch to GenCat-only coverage in 2023–2024 may affect comparisons.

Design follows the notes on geography and comparison (pp. 30–31, 60, 66–67). No downloads. Run all cells from the project folder.

### Period counts
Count fires on a common set of comarca boundaries and fill absent combinations with zero. The map shares use only fires with a known comarca.

In [ ]:
import pandas as pd
import altair as alt
from IPython.display import display

from comarca_maps import comarca_areas_hectares, comarca_base_map, load_comarca_boundaries, mercator

fires = pd.read_csv(
    "Clean data/CleanData.csv",
    parse_dates=["data_incendi"],
    dtype={"codi_comarca": "string"},
)
years = fires.data_incendi.dt.year
fires = fires.loc[years.between(1998, 2024)].copy()
fires["code"] = fires.codi_comarca

periods = ["1998–2006", "2007–2015", "2016–2024"]
fires["period"] = pd.cut(fires.data_incendi.dt.year, bins=[1997, 2006, 2015, 2024], labels=periods)

boundaries = load_comarca_boundaries(mercator)

codes = [f["properties"]["CODICOMAR"] for f in boundaries["features"]]
period_and_comarca = ["period", "code"]
all_combinations = pd.MultiIndex.from_product([periods, codes], names=period_and_comarca)
counts = (
    fires.groupby(period_and_comarca, observed=True).size()
    .reindex(all_combinations, fill_value=0)
    .reset_index(name="fires")
)
counts["share"] = counts.fires / counts.groupby("period").fires.transform("sum")
print("Excluded unknown comarca by period:", fires[fires.code.isna()].groupby("period", observed=False).size().to_dict())

def show_chart(chart):
    spec = chart.to_dict()
    spec["$schema"] = "https://vega.github.io/schema/vega-lite/v5.json"
    display({"application/vnd.vegalite.v5+json": spec, "text/html": chart.to_html()}, raw=True)


In [ ]:
share_scale = alt.Scale(domain=[0, counts.share.max()], scheme="reds")
maps = []
for period in periods:
    period_counts = counts.loc[counts.period.eq(period)]
    view = comarca_base_map(boundaries, stroke_width=0.5).transform_lookup(
        lookup="properties.CODICOMAR", from_=alt.LookupData(period_counts, "code", ["fires", "share"])
    ).encode(
        color=alt.Color("share:Q", title="Share of located fires", scale=share_scale, legend=alt.Legend(format=".0%", orient="bottom")),
        tooltip=[alt.Tooltip("properties.NOMCOMAR:N", title="Comarca"), "fires:Q", alt.Tooltip("share:Q", title="Share", format=".2%")]
    ).project(type="identity", reflectY=True).properties(width=280, height=310, title=period)
    maps.append(view)
chart = alt.hconcat(*maps).resolve_scale(color="shared").properties(title="Q4 · Where fires were concentrated (share within each period)").configure_view(stroke=None)
show_chart(chart)


## Q4 · Share of all period fires per hectare
Divide each comarca’s fire count by its area in hectares and by Catalonia’s total recorded fire count for the same period. The period total includes fires without a known comarca.

Comarca areas come from Idescat (2024) and are held fixed across periods. All three maps share a linear numeric scale with logarithmic colour contrast.

The legend uses parts per million (ppm) per hectare. One ppm equals 0.0001%.

In [ ]:
comarca_hectares = comarca_areas_hectares(boundaries)
density_counts = counts.copy()
density_counts["area_ha"] = density_counts.code.map(comarca_hectares)
assert density_counts.area_ha.notna().all() and density_counts.area_ha.gt(0).all(), "Missing or invalid comarca area"
period_totals = fires.groupby("period", observed=True).size()
density_counts["period_total"] = density_counts.period.map(period_totals).astype("int64")
density_counts["share_per_ha"] = density_counts.fires / (density_counts.area_ha * density_counts.period_total)

from math import log1p
density_max = float(density_counts.share_per_ha.max())
density_colour_values = [density_max * i / 100 for i in range(101)]
density_colour_range = []
for i in range(101):
    strength = log1p(9 * i / 100) / log1p(9)
    density_colour_range.append("#" + "".join(
        f"{round(light + strength * (dark - light)):02x}"
        for light, dark in zip((255, 245, 240), (103, 0, 13))))

density_maps = []
for period in periods:
    period_density = density_counts.loc[density_counts.period.eq(period)]
    density_view = comarca_base_map(boundaries, stroke_width=0.5).transform_lookup(
        lookup="properties.CODICOMAR",
        from_=alt.LookupData(period_density, "code", ["fires", "share", "area_ha", "period_total", "share_per_ha"])
    ).transform_calculate(share_ppm_per_ha="datum.share_per_ha * 1e6").encode(
        color=alt.Color("share_per_ha:Q", title="Share of all period fires per hectare", scale=alt.Scale(type="linear", domain=density_colour_values, range=density_colour_range), legend=alt.Legend(title="Share of period fires (ppm per hectare)", labelExpr="format(datum.value * 1e6, '.3~g')", orient="bottom")),
        tooltip=[alt.Tooltip("properties.NOMCOMAR:N", title="Comarca"),
                 alt.Tooltip("fires:Q", title="Comarca fires in period", format=",.0f"),
                 alt.Tooltip("period_total:Q", title="All fires in period", format=",.0f"),
                 alt.Tooltip("share:Q", title="Share of located fires", format=".2%"),
                 alt.Tooltip("area_ha:Q", title="Comarca area (ha)", format=",.0f"),
                 alt.Tooltip("share_ppm_per_ha:Q", title="Share of period fires (ppm per hectare)", format=".3~g")]
    ).project(type="identity", reflectY=True).properties(width=280, height=310, title=period)
    density_maps.append(density_view)
density_chart = alt.hconcat(*density_maps).resolve_scale(color="shared").properties(
    title="Q4 · Share of all period fires per hectare by comarca"
)
show_chart(density_chart.configure_view(stroke=None))


## Q4 · How the seasonal distribution changed
Rows are years; columns are months. **Darker red means a larger percentage of that year’s fires occurred in that month.** Read across a row to find the peak season; read down a column to see whether that month became more prominent. This complements the maps with temporal distribution, rather than repeating Q3’s annual totals.

Each year sums to 100%, so high-count years do not dominate. Hover for the monthly count, annual total and percentage. All merged fires from 1998–2024 are included, even those without a known comarca. Zero-count months are explicitly included. Percentages describe seasonal concentration, not increasing fire counts or daily risk; months have different lengths. Coverage changes to GenCat-only in 2023–2024.

The course notes support heatmaps for two ordered dimensions and pattern detection (pp. 30, 81); a single sequential red scale and chronological ordering make the matrix comparable. A chart with 27 overlapping yearly lines would be cluttered.


In [ ]:
year = fires.data_incendi.dt.year.rename("year")
month = fires.data_incendi.dt.month.rename("month")
all_months = pd.MultiIndex.from_product(
    [range(1998, 2025), range(1, 13)],
    names=["year", "month"],
)
monthly = (
    fires.groupby([year, month]).size()
    .reindex(all_months, fill_value=0)
    .reset_index(name="fires")
)
monthly["total"] = monthly.groupby("year").fires.transform("sum")
monthly["share"] = monthly.fires / monthly.total


In [ ]:
seasonality = alt.Chart(monthly).mark_rect(stroke="white", strokeWidth=0.5).encode(
    x=alt.X("month:O", title="Month", axis=alt.Axis(labelAngle=0, labelExpr="['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec'][datum.value - 1]")),
    y=alt.Y("year:O", title="Year", sort="ascending"),
    color=alt.Color("share:Q", title="Share of annual fires", scale=alt.Scale(scheme="reds", domain=[0, monthly.share.max()]), legend=alt.Legend(format=".0%")),
    tooltip=["year:O", "month:O", "fires:Q", alt.Tooltip("total:Q", title="Annual fires"), alt.Tooltip("share:Q", title="Share", format=".1%")]
).properties(width=650, height=540, title="Q4 · When fires occurred: changing seasonality (1998–2024)")
show_chart(seasonality)


## Final view

The share of all period fires per hectare and the seasonal heatmap use separate colour scales.

In [ ]:
final_view = (
    alt.vconcat(density_chart, seasonality)
    .resolve_scale(color="independent")
    .configure_view(stroke=None)
)
final_view